# Generative inference (OpenAI-compatible vLLM)

Workshop defaults: **`kserve-workshop`** project, **`granite-3-1-8b-instruct`** deployment.

1. Set **`INFERENCE_BASE_URL`** to the HTTPS route (no trailing slash), from OpenShift AI model deployment or `oc get inferenceservice`.
2. Set **`BEARER_TOKEN`** from Secret **`granite-3-1-8b-instruct-sa`** → key **`token`** (OpenShift console **Workloads → Secrets**).
3. Set **`MODEL_NAME`** to match an **`id`** returned by **`GET /v1/models`** (run the next cell first). If chat returns **model not found**, your name does not match what vLLM registered—copy the **`id`** exactly from that response (often the same as **`--served-model-name`** in `configs/samples/model-deploy/vllm-servingruntime.yaml`).

In [ ]:
import json
import urllib3
import requests

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- edit these ---
INFERENCE_BASE_URL = "https://REPLACE-with-your-route-host"
BEARER_TOKEN = "REPLACE-with-token-from-secret-granite-3-1-8b-instruct-sa"
# OpenAI "model" field must match a listed id from GET /v1/models (default matches workshop ServingRuntime)
MODEL_NAME = "granite-3-1-8b-instruct"

HEADERS = {
    "Authorization": f"Bearer {BEARER_TOKEN}",
    "Content-Type": "application/json",
}

SESSION = requests.Session()
SESSION.verify = False  # lab only; use proper CA trust in production


In [ ]:
r = SESSION.get(f"{INFERENCE_BASE_URL}/v1/models", headers=HEADERS, timeout=120)
print("status", r.status_code)
print(r.text[:4000])
if r.ok:
    payload = r.json()
    ids = [m.get("id") for m in payload.get("data", []) if m.get("id")]
    print("\nUse this exact string as MODEL_NAME in the previous cell if chat fails:")
    for i in ids:
        print("  ", repr(i))


In [ ]:
body = {
    "model": MODEL_NAME,
    "messages": [{"role": "user", "content": "Say hello in one short sentence."}],
    "max_tokens": 64,
}
r = SESSION.post(
    f"{INFERENCE_BASE_URL}/v1/chat/completions",
    headers=HEADERS,
    data=json.dumps(body),
    timeout=300,
)
print(r.status_code)
print(r.text)
